In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import distance

In [3]:
class KMeans:
    def __init__(self):
        self.centroids = []

    def _init_centroids(self, X, k):
        return X[np.random.choice(X.shape[0],k, replace=False)]

    def _assign_labels(self, X, centroids):
        D = distance.cdist(X, centroids)
        return np.argmin(D, axis=1)

    def _update_centroids(self, X, labels, K):
        centroids = np.zeros((K, X.shape[1]))
        for k in range(K):
            Xk = X[labels == k] # labels==k,:
            if len(Xk) == 0:                         #  xử lý cluster rỗng
                centroids[k] = self.centroids[-1][k]
            else:
                centroids[k] = np.mean(Xk, axis =0) # centroids[k,:]
        return centroids

    def has_converged(self, centroids, new_centroids):
        # return (set([tuple(a) for a in centroids]) == set([tuple(a) for a in new_centroids]))
        return np.allclose(
        np.sort(centroids, axis=0),
        np.sort(new_centroids, axis=0),
        atol=1e-4
    )
    def fit(self, X, K):
        current_centroid = self._init_centroids(X,K)
        self.centroids.append(current_centroid)
        # labels = []
        it = 0
        while True:
            #labels.append(self._assign_labels(X, current_centroid))

            # E-step
            labels = self._assign_labels(X,current_centroid)

            # M-step
            new_centroids = self._update_centroids(X, labels, K)
            if self.has_converged(current_centroid, new_centroids):
                break
            current_centroid = new_centroids
            self.centroids.append(new_centroids)
            it += 1

        return labels, it



###Assignment 1

In [4]:
means = [[2,2], [8,3],[3,6]]
cov = [[1,0], [0,1]]
N = 200

# Mỗi dòng sinh ra array shape (200, 2)
X0 = np.random.multivariate_normal(means[0], cov, N)
X1 = np.random.multivariate_normal(means[1], cov, N)
X2 = np.random.multivariate_normal(means[2], cov, N)

X = np.concatenate((X0, X1, X2), axis = 0)
K = 3

model = KMeans()
labels, it = model.fit(X,K)


print(f"Số vòng lặp: {it}")
print(f"Centroids tìm được:\n{model.centroids[-1]}")
print(f"Centroids thật:\n{means}")

Số vòng lặp: 3
Centroids tìm được:
[[2.05946643 1.91581722]
 [7.88314355 3.0770218 ]
 [3.1928625  5.88577144]]
Centroids thật:
[[2, 2], [8, 3], [3, 6]]


Nhận xét về ảnh hưởng của random initialization đến K-means
1. Ảnh hưởng đến tốc độ hội tụ: Việc khởi tạo centroids ngẫu nhiên ảnh hưởng trực tiếp đến số vòng lặp cần thiết để hội tụ. Nếu các centroids ban đầu được chọn gần với tâm thật của các cluster, thuật toán sẽ hội tụ nhanh hơn. Ngược lại, nếu chọn ở vị trí bất lợi, cần nhiều vòng lặp hơn.

2. Ảnh hưởng đến kết quả cuối
K-means chỉ đảm bảo hội tụ đến local optimum, không đảm bảo global optimum. Điều này có nghĩa là:
Nếu 2 centroids ban đầu cùng nằm trong 1 cluster → cluster còn lại bị chia sai
Kết quả cuối cùng phụ thuộc vào điểm khởi tạo, không phải lúc nào cũng đúng

### Assignment 2

In [6]:
means = [[2,2], [8,3],[3,6]]
cov = [[1,0], [0,1]]

X0 = np.random.multivariate_normal(means[0], cov, 1200)
X1 = np.random.multivariate_normal(means[1], cov, 200)
X2 = np.random.multivariate_normal(means[2], cov, 1000)

X = np.concatenate((X0, X1, X2), axis = 0)
K = 3

model = KMeans()
labels, it = model.fit(X,K)

print(f"Số vòng lặp: {it}")
print(f"Centroids tìm được:\n{model.centroids[-1]}")
print(f"Centroids thật:\n{means}")


Số vòng lặp: 5
Centroids tìm được:
[[8.02653005 3.05999009]
 [2.00624734 2.00389088]
 [3.02470749 6.02817312]]
Centroids thật:
[[2, 2], [8, 3], [3, 6]]


Nhận xét về ảnh hưởng của kích thước cluster không đều đến K-means

* Kết quả vẫn chính xác khi cluster tách biệt về không gian

* Số vòng lặp dao động nhiều hơn so với dataset cân bằng

* Gây sai kết quả khi vừa không đều kích thước, vừa overlap

### Assignment 3

In [8]:
means = [[2,2], [8,3],[3,6]]
cov = [[[1,0], [0,1]], [[10,0], [0,1]]]
N = 200

# Mỗi dòng sinh ra array shape (200, 2)
X0 = np.random.multivariate_normal(means[0], cov[0], N)
X1 = np.random.multivariate_normal(means[1], cov[0], N)
X2 = np.random.multivariate_normal(means[2], cov[1], N)

X = np.concatenate((X0, X1, X2), axis = 0)
K = 3

model = KMeans()
labels, it = model.fit(X,K)

print(f"Số vòng lặp: {it}")
print(f"Centroids tìm được:\n{model.centroids[-1]}")
print(f"Centroids thật:\n{means}")

Số vòng lặp: 8
Centroids tìm được:
[[7.92137059 3.6098802 ]
 [2.07893166 2.00613953]
 [1.61862046 6.04503466]]
Centroids thật:
[[2, 2], [8, 3], [3, 6]]


Ảnh hưởng của covariance matrix lớn $\mathcal{N}((3, 6), \Sigma_2)$

* Centroids tìm được kém chính xác hơn
Số vòng lặp tăng và không ổn định (3 → 20 vòng)

* Cluster bị trải rộng gây overlap → ranh giới không rõ ràng

* K-means không phù hợp vì chỉ dùng khoảng cách Euclidean, không mô hình hóa được hình dạng elipse '''